# Quotient Graph Operations

This notebook demonstrates quotient graph operations using topologic_fast.

## What is a Quotient Graph?

A **quotient graph** is a graph where vertices are grouped based on some equivalence relation,
and each group becomes a single vertex in the quotient graph.

In architectural terms:
- If you have many rooms but want to analyze connectivity by **room type** (office, corridor, etc.)
- The quotient graph groups all offices into one vertex, all corridors into another, etc.
- Edges represent connections between room types

**Note**: This notebook uses topologic_fast's native `Graph.Quotient()` implementation.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import defaultdict

## 1. Create a Building Layout with Room Types

We'll create a simple office building with different room types.

In [ ]:
# Define room data: (x, y, width, length, name, type)
room_data = [
    # Ground floor
    (0, 0, 4, 3, 'Lobby', 'public'),
    (4, 0, 3, 3, 'Office 1', 'office'),
    (7, 0, 3, 3, 'Office 2', 'office'),
    (0, 3, 10, 2, 'Main Corridor', 'corridor'),
    (0, 5, 3, 3, 'Conference A', 'meeting'),
    (3, 5, 3, 3, 'Office 3', 'office'),
    (6, 5, 4, 3, 'Break Room', 'amenity'),
    (0, 8, 5, 2, 'North Corridor', 'corridor'),
    (5, 8, 5, 2, 'Storage', 'service'),
    (0, 10, 3, 3, 'Office 4', 'office'),
    (3, 10, 4, 3, 'Conference B', 'meeting'),
    (7, 10, 3, 3, 'Restroom', 'service'),
]

# Create cells for each room
rooms = []
room_names = []
room_types = []
floor_height = 3.0

for x, y, w, l, name, rtype in room_data:
    cell = tf.Cell.Box(x, y, 0, w, l, floor_height)
    rooms.append(cell)
    room_names.append(name)
    room_types.append(rtype)

# Count room types
type_counts = defaultdict(int)
for rtype in room_types:
    type_counts[rtype] += 1

print(f"Created {len(rooms)} rooms:")
print(f"\nRoom Types:")
for rtype, count in sorted(type_counts.items()):
    print(f"  {rtype}: {count} rooms")

In [ ]:
# Create CellComplex and connectivity graph
building = tf.CellComplex.ByCells(rooms)
room_graph = tf.Graph.ByTopology(building)

print(f"Building Statistics:")
print(f"  Total rooms:      {len(rooms)}")
print(f"  Graph vertices:   {room_graph.Order()}")
print(f"  Connections:      {room_graph.Size()}")
print(f"  Graph density:    {room_graph.Density():.3f}")

## 2. Visualize the Room Graph

In [ ]:
# Color map for room types
color_map = {
    'public': '#87CEEB',     # Sky blue
    'office': '#90EE90',     # Light green
    'corridor': '#D3D3D3',   # Light gray
    'meeting': '#FFD700',    # Gold
    'amenity': '#FFDAB9',    # Peach
    'service': '#C0C0C0',    # Silver
}

def visualize_floor_plan(cellcomplex, room_names, room_types, graph=None):
    """Visualize a floor plan with optional connectivity graph."""
    fig = go.Figure()
    
    cells = cellcomplex.Cells()
    
    # Draw rooms
    for i, cell in enumerate(cells):
        faces = cell.Faces()
        color = color_map.get(room_types[i], '#FFFFFF')
        
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            z_coords = [c[2] for c in coords]
            
            # Bottom face (z = 0)
            if all(abs(z) < 0.01 for z in z_coords):
                x = [c[0] for c in coords] + [coords[0][0]]
                y = [c[1] for c in coords] + [coords[0][1]]
                
                fig.add_trace(go.Scatter(
                    x=x, y=y,
                    fill='toself',
                    fillcolor=color,
                    line=dict(color='black', width=2),
                    name=f"{room_names[i]} ({room_types[i]})",
                    hoverinfo='name'
                ))
                break
    
    if graph:
        # Draw graph edges
        graph_edges = graph.Edges()
        for edge in graph_edges:
            verts = edge.Vertices()
            if len(verts) == 2:
                p1 = verts[0].Coordinates()
                p2 = verts[1].Coordinates()
                fig.add_trace(go.Scatter(
                    x=[p1[0], p2[0]], y=[p1[1], p2[1]],
                    mode='lines',
                    line=dict(color='rgba(255,0,0,0.5)', width=3),
                    showlegend=False,
                    hoverinfo='skip'
                ))
        
        # Draw graph vertices
        graph_verts = graph.Vertices()
        fig.add_trace(go.Scatter(
            x=[v.X() for v in graph_verts],
            y=[v.Y() for v in graph_verts],
            mode='markers',
            marker=dict(size=10, color='red', line=dict(color='darkred', width=2)),
            name='Graph Nodes',
            hovertext=room_names,
            hoverinfo='text'
        ))
    
    fig.update_layout(
        xaxis=dict(title='X (m)', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y (m)'),
        width=800,
        height=700
    )
    
    return fig


fig_plan = visualize_floor_plan(building, room_names, room_types, room_graph)
fig_plan.update_layout(title='Office Floor Plan with Room Connectivity')
fig_plan.show()

# Compute quotient graph using topologic_fast's native implementation
# First, we need to assign partition labels to each vertex

# Get graph vertices
graph_vertices = room_graph.Vertices()

# Create a partition list (same order as vertices in graph)
# We match vertices by centroid position to room types
partition_labels = []
for gv in graph_vertices:
    gv_coords = gv.Coordinates()
    # Find which room this vertex corresponds to by matching centroid
    min_dist = float('inf')
    best_type = room_types[0]
    for i, cell in enumerate(building.Cells()):
        centroid = cell.CenterOfMass()
        dist = ((centroid[0] - gv_coords[0])**2 + 
                (centroid[1] - gv_coords[1])**2 + 
                (centroid[2] - gv_coords[2])**2)**0.5
        if dist < min_dist:
            min_dist = dist
            best_type = room_types[i]
    partition_labels.append(best_type)

# Compute quotient graph using native implementation
quotient_graph = room_graph.Quotient(partition_labels)

# Compute group info for visualization
unique_groups = sorted(set(partition_labels))
group_info = {}
for group in unique_groups:
    count = partition_labels.count(group)
    group_info[group] = {'name': group, 'count': count}

print("Quotient Graph (grouped by room type):")
print(f"  Groups (vertices): {quotient_graph.Order()}")
print(f"  Connections:       {quotient_graph.Size()}")
print(f"\nGroup Statistics:")
for group, info in group_info.items():
    print(f"  {group}: {info['count']} rooms")

In [ ]:
def compute_quotient_graph(graph, vertex_groups, group_names=None):
    """
    Compute the quotient graph by grouping vertices.
    
    Args:
        graph: The original graph
        vertex_groups: List assigning each vertex to a group (by index)
        group_names: Optional names for each group
    
    Returns:
        quotient_graph: The quotient graph
        group_info: Dictionary with group statistics
    """
    original_vertices = graph.Vertices()
    original_edges = graph.Edges()
    
    # Find unique groups
    unique_groups = list(set(vertex_groups))
    unique_groups.sort()
    group_to_idx = {g: i for i, g in enumerate(unique_groups)}
    
    # Calculate centroid for each group
    group_vertices = {g: [] for g in unique_groups}
    for i, v in enumerate(original_vertices):
        group = vertex_groups[i]
        group_vertices[group].append(v)
    
    # Create quotient vertices at group centroids
    quotient_verts = []
    group_info = {}
    
    for group in unique_groups:
        verts = group_vertices[group]
        cx = sum(v.X() for v in verts) / len(verts)
        cy = sum(v.Y() for v in verts) / len(verts)
        cz = sum(v.Z() for v in verts) / len(verts)
        
        quotient_verts.append(tf.Vertex.ByCoordinates(cx, cy, cz))
        
        name = group_names[group_to_idx[group]] if group_names else group
        group_info[group] = {
            'name': name,
            'count': len(verts),
            'centroid': (cx, cy, cz)
        }
    
    # Find edges between groups (and count connections)
    edge_weights = defaultdict(int)
    
    def get_vertex_group(vertex, tolerance=0.001):
        """Find which group a vertex belongs to."""
        vc = vertex.Coordinates()
        for i, ov in enumerate(original_vertices):
            oc = ov.Coordinates()
            if (abs(vc[0] - oc[0]) < tolerance and
                abs(vc[1] - oc[1]) < tolerance and
                abs(vc[2] - oc[2]) < tolerance):
                return vertex_groups[i]
        return None
    
    for edge in original_edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            g1 = get_vertex_group(verts[0])
            g2 = get_vertex_group(verts[1])
            
            if g1 is not None and g2 is not None and g1 != g2:
                # Sort to avoid duplicates
                edge_key = (min(g1, g2), max(g1, g2))
                edge_weights[edge_key] += 1
    
    # Create quotient edges
    quotient_edges = []
    edge_info = []
    
    for (g1, g2), weight in edge_weights.items():
        idx1 = group_to_idx[g1]
        idx2 = group_to_idx[g2]
        quotient_edges.append(
            tf.Edge.ByStartVertexEndVertex(quotient_verts[idx1], quotient_verts[idx2])
        )
        edge_info.append({
            'from': g1,
            'to': g2,
            'weight': weight
        })
    
    quotient_graph = tf.Graph.ByVerticesEdges(quotient_verts, quotient_edges)
    
    return quotient_graph, group_info, edge_info, unique_groups


# Compute quotient graph by room type
quotient_graph, group_info, edge_info, groups = compute_quotient_graph(
    room_graph, 
    room_types
)

print("Quotient Graph (grouped by room type):")
print(f"  Groups (vertices): {quotient_graph.Order()}")
print(f"  Connections:       {quotient_graph.Size()}")
print(f"\nGroup Statistics:")
for group, info in group_info.items():
    print(f"  {group}: {info['count']} rooms")

## 4. Visualize the Quotient Graph

In [ ]:
def visualize_quotient_graph(quotient_graph, group_info, edge_info, groups, color_map):
    """Visualize the quotient graph with group information."""
    fig = go.Figure()
    
    quotient_verts = quotient_graph.Vertices()
    quotient_edges = quotient_graph.Edges()
    
    # Draw edges with width based on weight
    for i, edge in enumerate(quotient_edges):
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            weight = edge_info[i]['weight'] if i < len(edge_info) else 1
            
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]], y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='darkgray', width=2 + weight * 2),
                showlegend=False,
                hovertext=f"Connections: {weight}",
                hoverinfo='text'
            ))
            
            # Add weight label
            mid_x = (p1[0] + p2[0]) / 2
            mid_y = (p1[1] + p2[1]) / 2
            fig.add_annotation(
                x=mid_x, y=mid_y,
                text=str(weight),
                showarrow=False,
                font=dict(size=12, color='red'),
                bgcolor='white'
            )
    
    # Draw vertices with size based on group count
    for i, (v, group) in enumerate(zip(quotient_verts, groups)):
        info = group_info[group]
        color = color_map.get(group, '#FFFFFF')
        size = 20 + info['count'] * 10
        
        fig.add_trace(go.Scatter(
            x=[v.X()], y=[v.Y()],
            mode='markers+text',
            marker=dict(size=size, color=color, line=dict(color='black', width=3)),
            text=[f"{group}\n({info['count']})"],
            textposition='middle center',
            textfont=dict(size=10),
            name=f"{group} ({info['count']} rooms)"
        ))
    
    fig.update_layout(
        title='Quotient Graph (Room Types)',
        xaxis=dict(title='X (m)', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y (m)'),
        width=700,
        height=600
    )
    
    return fig


fig_quotient = visualize_quotient_graph(quotient_graph, group_info, edge_info, groups, color_map)
fig_quotient.show()

## 5. Compare Original and Quotient Graphs Side by Side

In [ ]:
fig_compare = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'Original: {room_graph.Order()} rooms, {room_graph.Size()} connections',
        f'Quotient: {quotient_graph.Order()} types, {quotient_graph.Size()} connections'
    ]
)

# Draw original graph
orig_verts = room_graph.Vertices()
orig_edges = room_graph.Edges()

for edge in orig_edges:
    verts = edge.Vertices()
    if len(verts) == 2:
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig_compare.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color='gray', width=2),
            showlegend=False
        ), row=1, col=1)

# Color vertices by type
for i, v in enumerate(orig_verts):
    color = color_map.get(room_types[i], '#FFFFFF')
    fig_compare.add_trace(go.Scatter(
        x=[v.X()], y=[v.Y()],
        mode='markers',
        marker=dict(size=15, color=color, line=dict(color='black', width=1)),
        showlegend=False,
        hovertext=f"{room_names[i]} ({room_types[i]})",
        hoverinfo='text'
    ), row=1, col=1)

# Draw quotient graph
q_verts = quotient_graph.Vertices()
q_edges = quotient_graph.Edges()

for i, edge in enumerate(q_edges):
    verts = edge.Vertices()
    if len(verts) == 2:
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        weight = edge_info[i]['weight'] if i < len(edge_info) else 1
        fig_compare.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color='darkgray', width=1 + weight),
            showlegend=False
        ), row=1, col=2)

for i, (v, group) in enumerate(zip(q_verts, groups)):
    info = group_info[group]
    color = color_map.get(group, '#FFFFFF')
    size = 25 + info['count'] * 8
    
    fig_compare.add_trace(go.Scatter(
        x=[v.X()], y=[v.Y()],
        mode='markers+text',
        marker=dict(size=size, color=color, line=dict(color='black', width=2)),
        text=[f"{group}\n({info['count']})"],
        textposition='middle center',
        textfont=dict(size=9),
        showlegend=False
    ), row=1, col=2)

fig_compare.update_xaxes(scaleanchor="y", scaleratio=1)
fig_compare.update_layout(
    title='Original Room Graph vs Quotient Graph (by Room Type)',
    height=500,
    width=1200
)

fig_compare.show()

## 6. Connectivity Analysis

In [ ]:
print("=" * 60)
print("Room Type Connectivity Analysis")
print("=" * 60)

# Analyze connections between room types
print("\nConnections between room types:")
print("-" * 50)
print(f"{'From':<15} {'To':<15} {'Connections':<12}")
print("-" * 50)

for info in sorted(edge_info, key=lambda x: -x['weight']):
    print(f"{info['from']:<15} {info['to']:<15} {info['weight']:<12}")

# Calculate total connections per type
print("\nTotal connections per room type:")
print("-" * 40)

type_connections = defaultdict(int)
for info in edge_info:
    type_connections[info['from']] += info['weight']
    type_connections[info['to']] += info['weight']

for rtype, conns in sorted(type_connections.items(), key=lambda x: -x[1]):
    count = group_info[rtype]['count']
    avg = conns / count
    print(f"  {rtype:<12} {conns:>4} connections ({avg:.1f} per room)")

## 7. Weighted Quotient Graph Visualization

In [ ]:
# Create a more informative quotient graph visualization
fig_weighted = go.Figure()

# Calculate layout (circular arrangement)
import math

n_groups = len(groups)
radius = 4
angles = [2 * math.pi * i / n_groups for i in range(n_groups)]
positions = {group: (radius * math.cos(a), radius * math.sin(a)) 
             for group, a in zip(groups, angles)}

# Draw edges
max_weight = max(e['weight'] for e in edge_info) if edge_info else 1

for info in edge_info:
    p1 = positions[info['from']]
    p2 = positions[info['to']]
    width = 2 + (info['weight'] / max_weight) * 8
    
    fig_weighted.add_trace(go.Scatter(
        x=[p1[0], p2[0]], y=[p1[1], p2[1]],
        mode='lines',
        line=dict(color='rgba(100,100,100,0.6)', width=width),
        showlegend=False,
        hovertext=f"{info['from']} <-> {info['to']}: {info['weight']} connections",
        hoverinfo='text'
    ))
    
    # Weight label
    mid_x = (p1[0] + p2[0]) / 2
    mid_y = (p1[1] + p2[1]) / 2
    fig_weighted.add_annotation(
        x=mid_x, y=mid_y,
        text=str(info['weight']),
        showarrow=False,
        font=dict(size=12, color='darkred'),
        bgcolor='rgba(255,255,255,0.8)'
    )

# Draw vertices
max_count = max(info['count'] for info in group_info.values())

for group in groups:
    pos = positions[group]
    info = group_info[group]
    color = color_map.get(group, '#FFFFFF')
    size = 40 + (info['count'] / max_count) * 40
    
    fig_weighted.add_trace(go.Scatter(
        x=[pos[0]], y=[pos[1]],
        mode='markers+text',
        marker=dict(
            size=size, 
            color=color, 
            line=dict(color='black', width=3)
        ),
        text=[f"{group}\n({info['count']})"],
        textposition='middle center',
        textfont=dict(size=11),
        name=f"{group}: {info['count']} rooms"
    ))

fig_weighted.update_layout(
    title='Weighted Quotient Graph (Circular Layout)<br><sub>Node size = room count, Edge width = connection count</sub>',
    xaxis=dict(scaleanchor='y', scaleratio=1, showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    width=700,
    height=700
)

fig_weighted.show()

## 8. Quotient Graph Metrics

In [ ]:
print("\n" + "=" * 55)
print("Graph Metrics Comparison")
print("=" * 55)
print(f"{'Metric':<25} {'Original':<15} {'Quotient':<15}")
print("-" * 55)
print(f"{'Vertices':<25} {room_graph.Order():<15} {quotient_graph.Order():<15}")
print(f"{'Edges':<25} {room_graph.Size():<15} {quotient_graph.Size():<15}")
print(f"{'Density':<25} {room_graph.Density():<15.3f} {quotient_graph.Density():<15.3f}")
print(f"{'Diameter':<25} {room_graph.Diameter():<15} {quotient_graph.Diameter():<15}")
print(f"{'Max Degree':<25} {room_graph.MaximumDelta():<15} {quotient_graph.MaximumDelta():<15}")
print(f"{'Min Degree':<25} {room_graph.MinimumDelta():<15} {quotient_graph.MinimumDelta():<15}")
print(f"{'Is Complete':<25} {str(room_graph.IsComplete()):<15} {str(quotient_graph.IsComplete()):<15}")

# Calculate compression ratio
vertex_ratio = quotient_graph.Order() / room_graph.Order()
edge_ratio = quotient_graph.Size() / room_graph.Size() if room_graph.Size() > 0 else 0

print(f"\nCompression Ratios:")
print(f"  Vertex reduction: {(1-vertex_ratio)*100:.1f}%")
print(f"  Edge reduction:   {(1-edge_ratio)*100:.1f}%")

## 9. Alternative Groupings

We can create different quotient graphs by grouping rooms differently.

In [ ]:
# Group by function category (public vs private)
function_map = {
    'public': 'public',
    'office': 'private',
    'corridor': 'circulation',
    'meeting': 'shared',
    'amenity': 'shared',
    'service': 'service'
}

# Create function-based partition labels
function_labels = [function_map[t] for t in partition_labels]

# Compute quotient graph by function using native implementation
function_quotient = room_graph.Quotient(function_labels)

# Compute function group info
func_groups = sorted(set(function_labels))
func_info = {}
for group in func_groups:
    count = function_labels.count(group)
    func_info[group] = {'name': group, 'count': count}

print("Alternative Quotient Graph (by Function):")
print(f"  Groups:      {function_quotient.Order()}")
print(f"  Connections: {function_quotient.Size()}")
print(f"\nFunction Groups:")
for group, info in func_info.items():
    print(f"  {group}: {info['count']} rooms")

In [ ]:
# Compare both quotient graphs
fig_both = make_subplots(
    rows=1, cols=2,
    subplot_titles=['By Room Type', 'By Function']
)

function_colors = {
    'public': '#87CEEB',
    'private': '#90EE90',
    'circulation': '#D3D3D3',
    'shared': '#FFD700',
    'service': '#C0C0C0'
}

# Draw type quotient
for i, (v, group) in enumerate(zip(quotient_graph.Vertices(), groups)):
    info = group_info[group]
    color = color_map.get(group, '#FFFFFF')
    size = 25 + info['count'] * 8
    
    fig_both.add_trace(go.Scatter(
        x=[v.X()], y=[v.Y()],
        mode='markers+text',
        marker=dict(size=size, color=color, line=dict(color='black', width=2)),
        text=[group],
        textposition='top center',
        showlegend=False
    ), row=1, col=1)

for edge in quotient_graph.Edges():
    verts = edge.Vertices()
    if len(verts) == 2:
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig_both.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]],
            mode='lines', line=dict(color='gray', width=2),
            showlegend=False
        ), row=1, col=1)

# Draw function quotient
for i, (v, group) in enumerate(zip(function_quotient.Vertices(), func_groups)):
    info = func_info[group]
    color = function_colors.get(group, '#FFFFFF')
    size = 25 + info['count'] * 8
    
    fig_both.add_trace(go.Scatter(
        x=[v.X()], y=[v.Y()],
        mode='markers+text',
        marker=dict(size=size, color=color, line=dict(color='black', width=2)),
        text=[group],
        textposition='top center',
        showlegend=False
    ), row=1, col=2)

for edge in function_quotient.Edges():
    verts = edge.Vertices()
    if len(verts) == 2:
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig_both.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]],
            mode='lines', line=dict(color='gray', width=2),
            showlegend=False
        ), row=1, col=2)

fig_both.update_xaxes(scaleanchor="y", scaleratio=1)
fig_both.update_layout(
    title='Different Grouping Strategies for Quotient Graphs',
    height=500,
    width=1000
)

fig_both.show()

## Summary

This notebook demonstrated quotient graph operations using topologic_fast:

### Key Concepts

1. **Quotient Graph Definition**:
   - Vertices are grouped by an equivalence relation (e.g., room type)
   - Each group becomes a single vertex in the quotient graph
   - Edges represent connections between groups

2. **Weighted Quotient Graph**:
   - Edge weights count the number of connections between groups
   - Vertex sizes can represent group cardinality

3. **Applications**:
   - Simplifying complex building graphs
   - Analyzing connectivity patterns by room type
   - Understanding circulation patterns
   - Space syntax analysis at different scales

### topologic_fast Methods Used

- `tf.Graph.Quotient(partition)` - Compute quotient graph from partition labels
- `tf.Cell.Box()` - Create room cells
- `tf.CellComplex.ByCells()` - Combine rooms
- `tf.Graph.ByTopology()` - Create connectivity graph
- Graph metrics: `Order()`, `Size()`, `Density()`, etc.